In [ ]:
from speechlab.load import EmbeddingDataset
ds = EmbeddingDataset.from_hub(revision="v2.0")

In [ ]:
seg = ds.segments()
print(seg.shape) 
print(seg.columns.tolist())

In [ ]:
seg.groupby("split").session_id.nunique()

In [ ]:
X = ds.pooled("hubert_base", layer=6)
print(X.shape)
print(X[0][:10])

In [ ]:
Xs, y, sessions = ds.session_pool(X, weight_by_duration=True)

In [ ]:
print(Xs.shape, y.shape, y.mean())

In [ ]:
#E0: Can embeddings be a good predictor (simple LR, potensh RF) of PHQ-8 >= 10? 
import numpy as np

split_of_person = seg.groupby("session_id")["split"].first()
person_split = split_of_person.loc[sessions].values
print(np.unique(person_split, return_counts=True))

In [ ]:
is_train = person_split == "train"
is_dev = person_split == "dev"

X_train, y_train = Xs[is_train], y[is_train]
X_dev, y_dev = Xs[is_dev], y[is_dev]

In [ ]:
print(X_train.shape, y_train.mean())
print(X_dev.shape, y_dev.mean())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000))
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score

train_scores = model.predict_proba(X_train)[:, 1]
dev_scores = model.predict_proba(X_dev)[:, 1]

print("train AUC:", round(roc_auc_score(y_train, train_scores), 3))
print("dev AUC:", round(roc_auc_score(y_dev, dev_scores), 3))

In [ ]:
for C in [0.0001, 0.01, 1, 100]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    m.fit(X_train, y_train)
    tr = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    dv = roc_auc_score(y_dev,   m.predict_proba(X_dev)[:, 1])
    print(f"C={C:<7} train AUC={tr:.3f}  dev AUC={dv:.3f}")

In [ ]:
def dev_auc(X_segments, C=0.01):
    """Pool segment vectors to people, train on train people, return AUC on dev people."""
    Xp, yp, sess = ds.session_pool(X_segments, weight_by_duration=True)
    assert (sess == sessions).all()

    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    m.fit(Xp[is_train], yp[is_train])
    return roc_auc_score(yp[is_dev], m.predict_proba(Xp[is_dev])[:, 1])

print(dev_auc(ds.pooled("hubert_base", 6)))

In [ ]:
import pandas as pd

def get_X(model_name, layer):
    if model_name == "roberta_base":
        return ds.text("roberta_base", layer)    # text embeddings of the transcript
    return ds.pooled(model_name, layer)          # speech embeddings of the audio

rows = []
for model_name in ["hubert_base", "whisper_small", "roberta_base"]:
    for layer in range(13):
        auc = dev_auc(get_X(model_name, layer))
        rows.append({"model": model_name, "layer": layer, "dev_auc": auc})
        print(f"{model_name:14s} layer {layer:2d}  dev AUC {auc:.3f}")

results = pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt

for model_name, grp in results.groupby("model"):
    plt.plot(grp.layer, grp.dev_auc, marker="o", label=model_name)
plt.axhline(0.5, color="gray", linestyle="--", label="chance")
plt.xlabel("layer"); plt.ylabel("dev AUC (32 people)")
plt.legend(); plt.title("Depression probe by layer (train → dev)")
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold

is_trdev = is_train | is_dev       

def cv_scores(X_segments, C=0.01, n_repeats=5):
    Xp, yp, sess = ds.session_pool(X_segments, weight_by_duration=True)
    assert (sess == sessions).all()
    Xp, yp = Xp[is_trdev], yp[is_trdev]

    scores = np.zeros((n_repeats, len(yp)))
    for r in range(n_repeats):
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=r)
        for tr, te in folds.split(Xp, yp):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
            m.fit(Xp[tr], yp[tr])
            scores[r, te] = m.predict_proba(Xp[te])[:, 1]
    return yp, scores.mean(axis=0)

def bootstrap_ci(y_true, s, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue                     # AUC needs both classes in the resample
        aucs.append(roc_auc_score(y_true[i], s[i]))
    return np.percentile(aucs, [2.5, 97.5])

yv, sv = cv_scores(ds.pooled("hubert_base", 6))
print("HuBERT L6 CV AUC:", round(roc_auc_score(yv, sv), 3), "CI:", bootstrap_ci(yv, sv).round(2))


In [ ]:
rows = []
for model_name in ["hubert_base", "whisper_small", "roberta_base"]:
    for layer in range(13):
        yv, sv = cv_scores(get_X(model_name, layer))
        lo, hi = bootstrap_ci(yv, sv)
        auc = roc_auc_score(yv, sv)
        rows.append({"model": model_name, "layer": layer, "auc": auc, "lo": lo, "hi": hi})
        print(f"{model_name:14s} L{layer:2d}  AUC {auc:.3f}  [{lo:.2f}, {hi:.2f}]")
cv_results = pd.DataFrame(rows)


In [ ]:
for model_name, g in cv_results.groupby("model"):
    plt.plot(g.layer, g.auc, marker="o", label=model_name)
    plt.fill_between(g.layer, g.lo, g.hi, alpha=0.15)
plt.axhline(0.5, color="gray", linestyle="--", label="chance")
plt.xlabel("layer"); plt.ylabel("CV AUC (134 people, 95% CI)")
plt.legend(); plt.title("Depression probe by layer (5×5-fold CV)")
plt.show()